# AmsterdamUMCdb Exploration

## Capstone Goal
Predict successful weaning from mechanical ventilation using time-series deep learning models.

## Objectives
- Understand database structure
- Identify ventilation-related tables
- Find extubation events
- Explore physiological measurements
- Understand timestamps and patient identifiers

### Imports

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt

### Retrieving Google Project Id

In [ ]:
client = bigquery.Client()

print("Connected to BigQuery")

In [ ]:
# sets *your* project id
PROJECT_ID = "capstoneweaningprediction" #@param {type:"string"}

# Sets the default BigQuery dataset for accessing AmsterdamUMCdb

If you have received instructions to use a specific BigQuery instance, change the default settings here. Otherwise use these default values.

In [ ]:
# sets default dataset for AmsterdamUMCdb
DATASET_PROJECT_ID = 'amsterdamumcdb' #@param {type:"string"}
DATASET_ID = 'version1_5_0' #@param {type:"string"}
LOCATION = 'eu' #@param {type:"string"}

# Provide your credentials to access the AmsterdamUMCdb dataset on Google BigQuery
Authenticate your credentials with Google Cloud Platform and set your default Google Cloud Project ID as an environment variable for running query jobs.

1. Run the cell. The `Allow this notebook to access your Google credentials?` prompt appears. Select `Allow`.
2. In the `Sign in - Google Accounts` dialog, use the account you registered during the AmsterdamUMCdb application process and select `Allow` again.

In [ ]:
import os
from google.colab import auth

# all libraries check this environment variable, so set it:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

auth.authenticate_user()
print('Authenticated')

# Enable data table display

Colab includes the `google.colab.data_table` package that can be used to display Pandas dataframes as an interactive data table (default limits: `max_rows = 20000`, `max_columns = 20`). This is especially useful when exploring the  tables or dictionary from AmsterdamUMCdb. It can be enabled with:

In [ ]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable

# change default limits:
DataTable.max_columns = 50
DataTable.max_rows = 80000


## Set the default query job configuration for magics

In [ ]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
from google.cloud import bigquery

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
bigquery_magics.context.default_query_job_config = def_config

## Query the `person` table and copy the data to the `persons` Pandas dataframe:

The `person` table contains a record for each patient in AmsterdamUMCdb.

Since this is a relatively small table, it is acceptable to use `SELECT *`.

**Note**: Should an error occur while running the query, please see
the AmsterdamUMCdb BigQuery [Frequently Asked Questions](https://github.com/AmsterdamUMC/AmsterdamUMCdb/wiki/bigquery#faq).

In [ ]:
%%bigquery person
SELECT * FROM `amsterdamumcdb.version1_5_0.person`;


## Set the default query job configuration for google-cloud-bigquery client

In [ ]:
from google.cloud import bigquery

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

### List Tables

In [ ]:
query = f"""
SELECT table_name
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

tables = client.query(query).to_dataframe()

tables

Step 1 — Explore Measurement Table

In [ ]:
query = f"""
SELECT *
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE provider_id IS NOT NULL
LIMIT 10
"""

measurement_df = client.query(query).to_dataframe()

measurement_df.head()

In [ ]:
measurement_df['measurement_source_concept_id'].unique()

### Inspect Columns

In [ ]:
measurement_df.columns

### Count total measurements

In [ ]:
measurement_df.shape

### Identify Most Common Measurements

In [ ]:
query = f"""
SELECT
    c.concept_name,
    COUNT(*) as frequency
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
ON m.measurement_concept_id = c.concept_id
WHERE m.provider_id IS NOT NULL
GROUP BY c.concept_name
ORDER BY frequency DESC
LIMIT 50
"""

common_measurements = client.query(query).to_dataframe()

common_measurements

Ventilator parameters and continuous vitals dominate the top frequencies, reflecting the high-frequency logging of mechanically ventilated ICU patients.Ventilator parameters and continuous vitals dominate the top frequencies, reflecting the high-frequency logging of mechanically ventilated ICU patients. Lab values like electrolytes and coagulation markers appear less frequently, as they're drawn at discrete intervals rather than monitored continuously.

### Search Respiratory Measurements

In [ ]:
query = f"""
SELECT
    DISTINCT concept_name
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE LOWER(concept_name) LIKE '%resp%'
"""

resp_measurements = client.query(query).to_dataframe()

resp_measurements

### Explore Timestamp Distribution

In [ ]:
query = f"""
SELECT
    measurement_datetime
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE provider_id = 40
LIMIT 100
"""

timestamp_dist = client.query(query).to_dataframe()

timestamp_dist

The timestamps are irregular, with readings clustered around recurring time-of-day slots that likely reflect shift-based charting rather than any fixed interval. This is expected in ICU data, where measurement frequency shifts with patient acuity — some days show multiple readings hours apart, others have much longer gaps.

### Measurement Frequency Per Patient

In [ ]:
query = f"""
SELECT
    person_id,
    COUNT(*) as measurement_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE provider_id is not null
GROUP BY person_id
ORDER BY measurement_count DESC
LIMIT 20
"""

measurement_per_person = client.query(query).to_dataframe()

measurement_per_person

There's a stark density gap even within the top 20 — person 9989 has over 238k measurements while person 4353 sits at 81k, nearly a 3x difference, pointing to significant variation in ICU stay length or monitoring intensity. This kind of spread means any model trained on these sequences will need to handle highly variable sequence lengths rather than assuming uniform patient coverage.

### Explore Missingness

In [ ]:
measurement_df.isnull().sum()

The missingness here is structural rather than random — fields like `value_as_number`, `unit_concept_id`, and `range_low/high` are consistently absent together, suggesting certain measurement types simply don't carry numeric values or units, which will push the decision toward exclusion or concept-level stratification before any filling strategy makes sense.

### Distribution of Physiological Variables

#### Heart-Rate Distribution

In [ ]:
query = f"""
SELECT
    value_as_number
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
ON m.measurement_concept_id = c.concept_id
WHERE m.measurement_concept_id = 3027018
AND (m.provider_id IS NOT NULL OR m.provider_id IS NULL)
AND value_as_number IS NOT NULL
AND value_as_number > 0
LIMIT 1000
"""

heart_dist = client.query(query).to_dataframe()
heart_dist

In [ ]:
plt.hist(heart_dist['value_as_number'])

The distribution centers around 80-110 bpm, which is expected for ICU patients who tend to run slightly tachycardic. A small tail extends toward 140+ bpm representing genuinely elevated rates, while the few values below 40 are worth flagging as potential outliers or brief artefacts from monitoring interruptions.

#### Respiratoy-Rate Distribution

In [ ]:
query = f"""
SELECT
    value_as_number
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
ON m.measurement_concept_id = c.concept_id
WHERE m.measurement_concept_id = 3024171
AND (m.provider_id IS NOT NULL OR m.provider_id IS NULL)
AND value_as_number IS NOT NULL
AND value_as_number > 0
LIMIT 1000
"""

rr_dist = client.query(query).to_dataframe()
rr_dist

In [ ]:
plt.hist(rr_dist['value_as_number'])

The distribution is heavily concentrated at exactly 10 breaths per minute with a small cluster at 60, which is not a natural physiological spread — this looks like default or manually entered values rather than continuously monitored data. The near-empty range between 10 and 60 confirms this concept is likely capturing spot-check or ventilator-set rates rather than true continuous respiratory monitoring in this datase

In [ ]:
query = f"""
SELECT concept_id, concept_name, vocabulary_id, standard_concept
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE LOWER(concept_name) LIKE '%respiratory rate%'
AND standard_concept = 'S'
"""

concepts = client.query(query).to_dataframe()
concepts

#### SpO2 Distribution

In [ ]:
# First check the concept ID
query = f"""
SELECT concept_id, concept_name, vocabulary_id, standard_concept
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE LOWER(concept_name) LIKE '%oxygen saturation%'
AND standard_concept = 'S'
"""
concepts = client.query(query).to_dataframe()
concepts

In [ ]:
query = f"""
SELECT
    value_as_number,
    unit_source_value,
    unit_concept_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
WHERE m.measurement_concept_id = 40762499
AND (m.provider_id IS NOT NULL OR m.provider_id IS NULL)
AND value_as_number IS NOT NULL
LIMIT 1000
"""

spo2_dist = client.query(query).to_dataframe()
spo2_dist

In [ ]:
plt.hist(spo2_dist['value_as_number'])

The distribution is heavily right-skewed with the vast majority of values clustering at 90-100%, which is exactly where you'd expect a well-managed ICU population to sit with supplemental oxygen support. The sparse values below 70% are worth flagging — they could be genuine desaturation events during critical moments, or sensor artefacts from poor probe contact.

In [ ]:
# Step 1: Pick a patient with many measurements
query = f"""
SELECT
    person_id,
    COUNT(*) as measurement_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE (provider_id IS NOT NULL OR provider_id IS NULL)
GROUP BY person_id
ORDER BY measurement_count DESC
LIMIT 1
"""
top_patient = client.query(query).to_dataframe()
patient_id = top_patient['person_id'].values[0]
print(f"Selected patient: {patient_id}")

# Step 2: Pull all measurements for that patient
query = f"""
SELECT
    measurement_datetime,
    value_as_number,
    c.concept_name
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
ON m.measurement_concept_id = c.concept_id
WHERE m.person_id = {patient_id}
AND (m.provider_id IS NOT NULL OR m.provider_id IS NULL)
AND value_as_number IS NOT NULL
ORDER BY measurement_datetime
"""
patient_ts = client.query(query).to_dataframe()

# Step 3: Visualize time-series density
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

patient_ts['measurement_datetime'] = pd.to_datetime(patient_ts['measurement_datetime'])

# Get top 20 most frequent concepts for readability
top_concepts = patient_ts['concept_name'].value_counts().head(20).index
filtered_ts = patient_ts[patient_ts['concept_name'].isin(top_concepts)]

# Shorten long concept names
filtered_ts = filtered_ts.copy()
filtered_ts['concept_short'] = filtered_ts['concept_name'].str[:40]

plt.figure(figsize=(18, 10))
concepts = filtered_ts['concept_short'].unique()
concept_map = {c: i for i, c in enumerate(concepts)}

plt.scatter(
    filtered_ts['measurement_datetime'],
    filtered_ts['concept_short'].map(concept_map),
    s=1, alpha=0.3, color='steelblue'
)

plt.yticks(range(len(concepts)), concepts, fontsize=9)
plt.xlabel('Time', fontsize=12)
plt.title(f'Time-Series Density for Patient {patient_id} (Top 20 Concepts)', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Step 4: Check gaps between consecutive measurements
patient_ts_sorted = patient_ts.sort_values('measurement_datetime')
patient_ts_sorted['gap_minutes'] = patient_ts_sorted['measurement_datetime'].diff().dt.total_seconds() / 60
print("Gap statistics (minutes):")
print(patient_ts_sorted['gap_minutes'].describe())
print(f"\nLargest gaps (hours):")
print((patient_ts_sorted['gap_minutes'].nlargest(5) / 60).round(1))

The plot reveals two distinct ICU admission periods — a dense monitoring block from January to February 2013, then a gap, followed by another from April to May…The plot reveals two distinct ICU admission periods — a dense monitoring block from January to February 2013, then a gap, followed by another from April to May 2013, suggesting either a readmission or a transfer. The gap statistics confirm this, with a maximum gap of 501 hours (~3 weeks) and a median of zero, meaning when this patient was being monitored the measurements were nearly continuous, but the inter-admission silence is substantial — exactly the kind of discontinuity that needs to be handled carefully before feeding sequences into an LSTM/GRU.